In [2]:
# Importing essential libraries

import os
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import pylab as pl
from glob import glob

import numpy as np
import tensorflow as tf
# tf.get_logger().setLevel('INFO')
import pickle as pkl

tf.autograph.set_verbosity(0)

from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from sklearn.metrics import top_k_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

from tensorflow.keras.layers import Input, Dense, Layer, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.losses import BinaryCrossentropy, CategoricalCrossentropy, MeanSquaredError
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model

from typing import List, Tuple
from tqdm import tqdm

np.random.seed(123)
tf.random.set_seed(1234)

import warnings
warnings.filterwarnings('ignore')
import more_itertools as mit

In [ ]:
# inputs = Input((224, 224, 3))
# conv1 = Conv2D(32, kernel_size = 4, activation = tf.nn.relu, name = 'fc_CS1')(inputs)
# pool1 = MaxPooling2D(pool_size = (2, 2))(conv1)
# conv2 = Conv2D(16, kernel_size = 4, activation = tf.nn.relu, name = 'fc_CS2')(pool1)
# pool2 = MaxPooling2D(pool_size = (2, 2))(conv2)
# flat = Flatten()(pool2)
# net = Dense(units=400, activation = tf.nn.relu, name="fc_GS1")(flat)
# net = Dense(units=400, activation = tf.nn.relu, name="fc_GS2")(net)        
# net = Dense(units=400, activation = tf.nn.relu, name="fc_GS3")(net)
# net = Dense(units=400, activation = tf.nn.relu, name="fc_GS4")(net)

# feature_T = Dense(units = 512, activation = tf.nn.sigmoid, name = "feature_T")(net)
# G_T = Model(inputs = inputs, outputs = feature_T, name = "Generator_T")

In [ ]:
# G_T.summary()

In [3]:
base_dir = '../input/hetrogenous-balanced-dataset/hetrogenous2'
print(os.listdir(base_dir))

['source', 'target']


In [4]:
source_dir = os.path.join(base_dir, 'source')
target_dir = os.path.join(base_dir, 'target')
conv_model_weights_dir = '/kaggle/input/weights/weights/resnet_50_224x224.h5'

In [5]:
print(f"no. of images in source : {len(glob(source_dir + '/**/**/*'))}")
print(f"no. of images in target : {len(glob(target_dir + '/**/**/*'))}")

source_target_data_distribution_ratio = round(len(glob(source_dir + '/**/**/*')) / len(glob(target_dir + '/**/**/*')), 2)
print(source_target_data_distribution_ratio)

no. of images in source : 5814
no. of images in target : 1099
5.29


In [6]:
class MalwareDetection:
    """
        This class is an inference for trained models on malware images data
    """
    def __init__(self, model_path : str, optimizer, loss_fn : str, 
                         metrics : List[str], input_shape : Tuple):
        self.model_path = model_path
        self.optimizer = optimizer
        self.loss = loss_fn
        self.metrics = metrics
        self.input_shape = input_shape
        self.model = load_model(self.model_path)
        self.classes = ["benign", "malicious"]

    def load_image(self, img_path):
        image = cv2.imread(img_path)
        image_resized = cv2.resize(image, self.input_shape)
        image = np.expand_dims(image_resized, axis = 0)
        print("image load successfully")
#         print(img_path)
        return image

    def check_malware_image(self, img_path):
        img = self.load_image(img_path)
        out = self.model.predict(img)[0]
        pred = self.classes[np.argmax(list(out))]
        return f"predicted class is : {pred}"

    def get_embeddings(self, img_path):
        """
            This method is used to get second last layers embeddings
        """
        img = self.load_image(img_path)
        extractor = Model(inputs = model.inputs,
                          outputs = [model.layers[-2].output])
        emb = extractor(img)
        return emb

In [9]:
class MalwareImageGAN:
    def __init__(self, source_images_dir : str, target_images_dir : str,\
                    input_shape : Tuple, conv_model_path : str,\
                    n_steps : int = 2000, batch_size : int = 4):
        """This class is a GAN architecture for detecting and generating
            malware images

        Args:
            source_images_dir (str): directory for source images
            target_images_dir (str): directory for target images
            input_shape (tuple) : input shape of images
            conv_model_path (str): path for pretrained model on malware images
        """

        self.source_images_dir = source_images_dir
        self.target_images_dir = target_images_dir

        self.source_train_images = os.path.join(self.source_images_dir, 'train')
        self.source_test_images = os.path.join(self.source_images_dir, 'test')

        self.target_train_images = os.path.join(self.target_images_dir, 'train')
        self.target_test_images = os.path.join(self.target_images_dir, 'test')

        self.input_shape = input_shape
        self.conv_model_path = conv_model_path
        self.n_classes = 2

        self.latent_dim = 512
        self.optimizer = Adam(0.0002, 0.5)  # Adam(1e-5)
        self.batch_size = batch_size
        self.n_steps = n_steps
        self.class_mapper = {
            0 : [1, 0], 1 : [0, 1],
        }


    def conv_model(self):
        malwareConvNet = MalwareDetection(model_path = self.conv_model_path,
                          optimizer = Adam(learning_rate = 0.001),
                          loss_fn = 'sparse_categorical_crossentropy',
                          metrics = ['accuracy'],
                          input_shape = self.input_shape)
        return malwareConvNet.model

    def build_generator_S(self):
        print("\n== Build Generator S...")
        model = self.conv_model()
        G_S = Model(inputs = model.inputs, \
            outputs = [model.layers[-2].output], name = "Generator_S")
        return G_S

    def build_generator_T(self):
        print("\n== Build Generator T...")
        model = self.conv_model()
        G_T = Model(inputs = model.inputs, \
            outputs = [model.layers[-2].output], name = "Generator_T")
            # [e1, e2, ....., en] 1 X 512
        return G_T

    def build_generator(self):
        print("\n== Build Generator...")

        inputs = Input(self.latent_dim)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G1")(inputs)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G2")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G3")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G4")(net)

        DIrep = Dense(units = self.latent_dim, activation = tf.nn.sigmoid, name = "DIrep")(net)
        G = Model(inputs = inputs, outputs = DIrep, name = "Generator")

        #Classifier
        inputs = Input(DIrep.shape)
        net = Dense(units = 800, activation = tf.nn.relu, name = "fc_C0")(inputs)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_C1")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_C2")(net)
        net = Dense(units = 200, activation = tf.nn.relu, name = "fc_C3")(net)
        net = Dense(units = 200, activation = tf.nn.relu, name = "fc_C4")(net)
        net = Dense(units = self.n_classes, activation = tf.nn.softmax, name = "C")(net)
        C = Model(inputs = inputs, outputs = net, name = "Classifier")

        return G, C

    def build_disciminator(self):
        print("\n== Build Discriminator...")

        inputs = Input(self.latent_dim)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D1")(inputs)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D2")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D3")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D4")(net)

        net = Dense(units = self.n_classes, activation = tf.nn.softmax, name = "D")(net)
        D = Model(inputs = inputs, outputs = net, name = "Discriminator")
        return D

    def create_image_tensor(self):
        """
            This method is used to create image tensors for testing folder
        """
        S_test_images = []
        S_test_labels = []
        T_test_images = []
        T_test_labels = []
        normal_images_source = glob(self.source_test_images + '/normal/*')
        normal_images_target = glob(self.target_test_images + '/normal/*')
        malware_images_source = glob(self.source_test_images + '/attack/*')
        malware_images_target = glob(self.target_test_images + '/attack/*')
#         normal_images_source = []
#         normal_images_target = []
        for cl, cat in enumerate([normal_images_source, malware_images_source]):
            for i, im in enumerate(cat):
                if i > 50:
                    break
                label = cl
                img = tf.io.read_file(im)
                tensor = tf.io.decode_image(img, channels = 3, dtype = tf.dtypes.float32)
                tensor = tf.image.resize(tensor, list(self.input_shape))
                S_test_images.append(tensor)
                S_test_labels.append(label)

        for cl, cat in enumerate([normal_images_target, malware_images_target]):
            for i, im in enumerate(cat):
                if i > 50:
                    break
                label = cl
                img = tf.io.read_file(im)
                tensor = tf.io.decode_image(img, channels = 3, dtype = tf.dtypes.float32)
                tensor = tf.image.resize(tensor, list(self.input_shape))
                T_test_images.append(tensor)
                T_test_labels.append(label)

        S_test_images = tf.convert_to_tensor(S_test_images)
        S_test_labels = tf.convert_to_tensor(S_test_labels)
        T_test_images = tf.convert_to_tensor(T_test_images)
        T_test_labels = tf.convert_to_tensor(T_test_labels)
        return S_test_images, S_test_labels, T_test_images, T_test_labels


    def d_loss(self, yhat_source, yhat_target):
        y_source = np.tile([1,0], (yhat_source.shape[0], 1))
        y_target = np.tile([0,1], (yhat_target.shape[0], 1))

        bce = CategoricalCrossentropy(from_logits = False)
        return bce(y_source, yhat_source) + bce(y_target, yhat_target)

    def g_loss(self, yhat_source, yhat_target):
        #[0,1]
        #[0,1]
        # ...

        y_source = np.tile([0,1], (yhat_source.shape[0], 1))
        y_target = np.tile([1,0], (yhat_target.shape[0], 1))

        bce = CategoricalCrossentropy(from_logits = False)
        return bce(y_source, yhat_source) + bce(y_target, yhat_target)

    def c_loss(self, yhat_class_source, yhat_class_target, y_source, y_target):
        # source_weight = 0.5
        # target_weight = 1
        bce = CategoricalCrossentropy(from_logits = False)
        # return (source_weight*bce(y_source, yhat_class_source) + target_weight* bce(y_target, yhat_class_target))/(source_weight + target_weight)
        return bce(y_source, yhat_class_source) + bce(y_target, yhat_class_target) #weight-source. bce() + .../(ws+wtt)


    def train(self):
        D = self.build_disciminator()
        G_S = self.build_generator_S()
        G_T = self.build_generator_T()
        G, C = self.build_generator()

        S_batches = tf.keras.preprocessing.image_dataset_from_directory(self.source_train_images,
                                                      seed = 123,
                                                      image_size = self.input_shape,
                                                      batch_size = int(self.batch_size * source_target_data_distribution_ratio * 0.5))

        T_batches = tf.keras.preprocessing.image_dataset_from_directory(self.target_train_images,
                                                      seed = 123,
                                                      image_size = self.input_shape,
                                                      batch_size = self.batch_size)

#         S_test_images, S_test_labels, T_test_images, T_test_labels = self.create_image_tensor()

        S_test_batches = tf.keras.preprocessing.image_dataset_from_directory(self.source_test_images,
                                                      seed = 123,
                                                      image_size = self.input_shape,
                                                      batch_size = 50)
    
        T_test_batches = tf.keras.preprocessing.image_dataset_from_directory(self.source_test_images,
                                                      seed = 123,
                                                      image_size = self.input_shape,
                                                      batch_size = 50)

        S_batches = iter(S_batches)
        T_batches = iter(T_batches)
        S_batches = mit.seekable(S_batches)
        T_batches = mit.seekable(T_batches)
        
        S_test_batches = iter(S_test_batches)
        T_test_batches = iter(T_test_batches)

        optimizer = self.optimizer

        g_loss_weight = 1
        c_loss_weight = 1

        print('====Loss Weights====')
        print('g_loss_weight: {0}'.format(g_loss_weight))
        print('c_loss_weight: {0}'.format(c_loss_weight))

        def _train_step(step):

            # Get a batch of source and target unlabeled samples
            try:
                x_batch_source, y_batch_source = next(S_batches)
            except:
                S_batches.seek(0)
                x_batch_source, y_batch_source = next(S_batches)
            try:
                x_batch_target, y_batch_target = next(T_batches)
            except:
                T_batches.seek(0)
                x_batch_target, y_batch_target = next(T_batches)
                
            #Create feature selections
            feature_S = G_S(x_batch_source)
            feature_T = G_T(x_batch_target)

            #Create domain invariant mapping using the Generator
            DIrep_source_samples = G(feature_S)
            DIrep_target_samples = G(feature_T)

            # Calculate the Domain loss
            with tf.GradientTape(persistent = True) as tape_disc:
                #Predict the domain using the discriminator
                yhat_source = D(DIrep_source_samples)
                yhat_target = D(DIrep_target_samples)
                
                # Compute D loss
                d_loss_value = self.d_loss(yhat_source, yhat_target)

            # Given loss, compute and apply gradient for discriminator:
            d_gradients = tape_disc.gradient(d_loss_value, D.trainable_variables)
            optimizer.apply_gradients(zip(d_gradients, D.trainable_variables))


            try:
                x_batch_source, y_batch_source = next(S_batches)
            except:
                S_batches.seek(0)
                x_batch_source, y_batch_source = next(S_batches)
            try:
                x_batch_target, y_batch_target = next(T_batches)
            except:
                T_batches.seek(0)
                x_batch_target, y_batch_target = next(T_batches)
            

            with tf.GradientTape(persistent = True) as tape_gen:

                #Create feature selections
                feature_S = G_S(x_batch_source)
                feature_T = G_T(x_batch_target)

                #Create domain invariant mapping using the Generator
                DIrep_source_samples = G(feature_S)
                DIrep_target_samples = G(feature_T)


                #Predict the domain using the discriminator
                yhat_source = D(DIrep_source_samples)
                yhat_target= D(DIrep_target_samples)

                #Predict the class of the samples

                class_pred_source = C(DIrep_source_samples)
                class_pred_target = C(DIrep_target_samples)
                
                # Compute G loss
                g_loss_value = self.g_loss(yhat_source, yhat_target)
                # Compute C loss

                y_batch_source = np.array(y_batch_source)
                y_batch_target = np.array(y_batch_target)

                
                try:
                    y_batch_source_dummy = [self.class_mapper[y_batch_source[i]] for i in range(len(y_batch_source))]
                    y_batch_target_dummy = [self.class_mapper[y_batch_target[i]] for i in range(len(y_batch_target))]
                except:
                    print(len(y_batch_source))
                    print(len(y_batch_target))

                y_batch_source = tf.Variable(y_batch_source_dummy, dtype = tf.float32)
                y_batch_target = tf.Variable(y_batch_target_dummy, dtype = tf.float32)
 
                c_loss_value = self.c_loss(class_pred_source, class_pred_target,
                                      y_batch_source, y_batch_target)


                combined_loss_value = (g_loss_weight * g_loss_value + c_loss_weight * c_loss_value) / (g_loss_weight + c_loss_weight)

            c_gradients = tape_gen.gradient(c_loss_value, C.trainable_variables)
            gs_gradients = tape_gen.gradient(combined_loss_value, G_S.trainable_variables)
            gt_gradients = tape_gen.gradient(combined_loss_value, G_T.trainable_variables)
            g_gradients = tape_gen.gradient(combined_loss_value, G.trainable_variables)

            optimizer.apply_gradients(zip(gs_gradients, G_S.trainable_variables))
            optimizer.apply_gradients(zip(gt_gradients, G_T.trainable_variables))
            optimizer.apply_gradients(zip(g_gradients, G.trainable_variables))
            optimizer.apply_gradients(zip(c_gradients, C.trainable_variables))

            return G_S, G_T, G, C, D, g_loss_value, c_loss_value, d_loss_value, combined_loss_value

        for step in range(1, self.n_steps):
            generator_S, generator_T, generator, classifier, discriminator, g_loss_value, c_loss_value, d_loss_value, combined_loss_value = _train_step(step)

            if (step % 50) == 0:
                
                x_test_batch_source, y_test_batch_source = next(S_test_batches)
                x_test_batch_target, y_test_batch_target = next(T_test_batches)
                test_source_gen = G(G_S(x_test_batch_source))
                test_target_gen = G(G_T(x_test_batch_target))

                source_pred = C(test_source_gen)
                target_pred = C(test_target_gen)

                print("RESULTS ON TEST SOURCE IMAGES :")
                print(classification_report(tf.math.argmax(source_pred, 1).numpy(), y_test_batch_source.numpy()))

                print("RESULTS ON TEST TARGET IMAGES :")
                print(classification_report(tf.math.argmax(target_pred, 1).numpy(), y_test_batch_target.numpy()))

                
                accuracy_source = accuracy_score(y_test_batch_source.numpy(), tf.math.argmax(source_pred, 1).numpy())
                accuracy_target = accuracy_score(y_test_batch_target.numpy(), tf.math.argmax(target_pred, 1).numpy())

                y_source_DI_test = generator(generator_S(x_test_batch_source))
                y_target_DI_test = generator(generator_T(x_test_batch_target))


                y_source_domain_pred = discriminator(y_source_DI_test).numpy().argmax(1)
                y_target_domain_pred = discriminator(y_target_DI_test).numpy().argmax(1)
                y_domain_pred = tf.concat([y_source_domain_pred, y_target_domain_pred], axis=0)
                #why it is 1,0?
                y_domain_source_real = np.array([1] * y_source_domain_pred.shape[0])
                y_domain_target_real = np.array([0] * y_target_domain_pred.shape[0])
                y_domain_real =  tf.concat([y_domain_source_real, y_domain_target_real], axis=0)
                # print((y_domain_pred.numpy() == 1).sum())
                # print((y_domain_real.numpy() == 1).sum())

                domain_pred_accuracy_source = accuracy_score(y_domain_source_real, y_source_domain_pred)
                domain_pred_accuracy_target = accuracy_score(y_domain_target_real, y_target_domain_pred)

                f1_source = f1_score(y_test_batch_source.numpy(), tf.math.argmax(source_pred, 1).numpy(), average = 'weighted')
                f1_target = f1_score(y_test_batch_target.numpy(), tf.math.argmax(target_pred, 1).numpy(), average = 'weighted')

                track_loss = '\nStep %4d ==>Comb_loss: %4.4f G_Loss: %4.4f C_Loss: %4.4f D_Loss: %4.4f \n Acc Source: %4.4f Acc Target: %4.4f F1 Source: %4.4f F1 Target: %4.4f \n Acc Domain Source: %4.4f  Acc Domain Target: %4.4f' % (
                                                            step, combined_loss_value, g_loss_value.numpy(), c_loss_value.numpy(), d_loss_value.numpy(), 
                                                            accuracy_source, accuracy_target, f1_source, f1_target,
                                                            domain_pred_accuracy_source, domain_pred_accuracy_target)
                print(track_loss)

        print('Training ended')

In [10]:
malGAN = MalwareImageGAN(source_images_dir = source_dir, \
                            target_images_dir = target_dir, \
                            input_shape = (224, 224), \
                            conv_model_path = conv_model_weights_dir,
                            n_steps = 1000,
                            batch_size = 16)


try:
    malGAN.train()
except Exception as e:
    print(str(e))
    import traceback
    traceback.print_tb(e.__traceback__)


== Build Discriminator...

== Build Generator S...

== Build Generator T...

== Build Generator...
Found 4864 files belonging to 2 classes.
Found 901 files belonging to 2 classes.
Found 950 files belonging to 2 classes.
Found 950 files belonging to 2 classes.
====Loss Weights====
g_loss_weight: 1
c_loss_weight: 1


Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup ca

RESULTS ON TEST SOURCE IMAGES :
              precision    recall  f1-score   support

           0       0.86      0.95      0.90        38
           1       0.75      0.50      0.60        12

    accuracy                           0.84        50
   macro avg       0.80      0.72      0.75        50
weighted avg       0.83      0.84      0.83        50

RESULTS ON TEST TARGET IMAGES :
              precision    recall  f1-score   support

           0       1.00      0.84      0.91        50
           1       0.00      0.00      0.00         0

    accuracy                           0.84        50
   macro avg       0.50      0.42      0.46        50
weighted avg       1.00      0.84      0.91        50


Step   50 ==>Comb_loss: 1.0693 G_Loss: 1.4286 C_Loss: 0.7100 D_Loss: 1.4132 
 Acc Source: 0.8400 Acc Target: 0.8400 F1 Source: 0.8520 F1 Target: 0.7670 
 Acc Domain Source: 0.2800  Acc Domain Target: 1.0000


Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup ca

RESULTS ON TEST SOURCE IMAGES :
              precision    recall  f1-score   support

           0       0.88      1.00      0.94        36
           1       1.00      0.64      0.78        14

    accuracy                           0.90        50
   macro avg       0.94      0.82      0.86        50
weighted avg       0.91      0.90      0.89        50

RESULTS ON TEST TARGET IMAGES :
              precision    recall  f1-score   support

           0       0.76      0.78      0.77        40
           1       0.00      0.00      0.00        10

    accuracy                           0.62        50
   macro avg       0.38      0.39      0.38        50
weighted avg       0.60      0.62      0.61        50


Step  100 ==>Comb_loss: 1.0688 G_Loss: 1.4176 C_Loss: 0.7200 D_Loss: 1.4177 
 Acc Source: 0.9000 Acc Target: 0.6200 F1 Source: 0.9076 F1 Target: 0.6277 
 Acc Domain Source: 1.0000  Acc Domain Target: 0.3600


...

RESULTS ON TEST SOURCE IMAGES :
              precision    recall  f1-score   support

           0       0.90      1.00      0.95        38
           1       1.00      0.67      0.80        12

    accuracy                           0.92        50
   macro avg       0.95      0.83      0.88        50
weighted avg       0.93      0.92      0.91        50

RESULTS ON TEST TARGET IMAGES :
              precision    recall  f1-score   support

           0       1.00      0.84      0.91        50
           1       0.00      0.00      0.00         0

    accuracy                           0.84        50
   macro avg       0.50      0.42      0.46        50
weighted avg       1.00      0.84      0.91        50


Step  400 ==>Comb_loss: 0.7316 G_Loss: 1.3847 C_Loss: 0.0784 D_Loss: 1.3740 
 Acc Source: 0.9200 Acc Target: 0.8400 F1 Source: 0.9260 F1 Target: 0.7670 
 Acc Domain Source: 0.1800  Acc Domain Target: 1.0000


Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup ca

RESULTS ON TEST SOURCE IMAGES :
              precision    recall  f1-score   support

           0       0.86      1.00      0.92        30
           1       1.00      0.75      0.86        20

    accuracy                           0.90        50
   macro avg       0.93      0.88      0.89        50
weighted avg       0.91      0.90      0.90        50

RESULTS ON TEST TARGET IMAGES :
              precision    recall  f1-score   support

           0       0.89      0.69      0.78        45
           1       0.07      0.20      0.10         5

    accuracy                           0.64        50
   macro avg       0.48      0.44      0.44        50
weighted avg       0.80      0.64      0.71        50


Step  450 ==>Comb_loss: 0.7454 G_Loss: 1.3800 C_Loss: 0.1107 D_Loss: 1.3934 
 Acc Source: 0.9000 Acc Target: 0.6400 F1 Source: 0.9033 F1 Target: 0.5725 
 Acc Domain Source: 0.3800  Acc Domain Target: 0.9200


Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup called...
Cleanup ca

RESULTS ON TEST SOURCE IMAGES :
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        43
           1       1.00      1.00      1.00         7

    accuracy                           1.00        50
   macro avg       1.00      1.00      1.00        50
weighted avg       1.00      1.00      1.00        50

RESULTS ON TEST TARGET IMAGES :
              precision    recall  f1-score   support

           0       0.93      0.85      0.89        47
           1       0.00      0.00      0.00         3

    accuracy                           0.80        50
   macro avg       0.47      0.43      0.44        50
weighted avg       0.87      0.80      0.84        50


Step  500 ==>Comb_loss: 0.7283 G_Loss: 1.3766 C_Loss: 0.0799 D_Loss: 1.3834 
 Acc Source: 1.0000 Acc Target: 0.8000 F1 Source: 1.0000 F1 Target: 0.7644 
 Acc Domain Source: 0.4000  Acc Domain Target: 0.9200
RESULTS ON TEST SOURCE IMAGES :
              precision    recall  f1-scor